<a href="https://colab.research.google.com/github/laurakeita/laura-portfolio/blob/main/MMM_meridian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install git+https://github.com/google/meridian.git altair

  Cloning https://github.com/google/meridian.git to /tmp/pip-req-build-jdmiojf8
  Running command git clone --filter=blob:none --quiet https://github.com/google/meridian.git /tmp/pip-req-build-jdmiojf8
  Resolved https://github.com/google/meridian.git to commit 036431ed36d37786c4b92bde6881f7281cc49755
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.3 MB/s eta 0:00:00
  Created wheel for google-meridian: filename=google_meridian-1.6.2-py3-none-any.whl size=804860 sha256=4cf1d1e0d1c43f81d66e3ec2a016872c72d81c4c2ab3f932fb2da7cbb420a77c
  Stored in directory: /tmp/pip-ephem-wheel-cache-nnudmyo4/wheels/a3/47/ca/62f337f613a98a4e2bd3793ada0c1752985e0f302e30d3c991
Successfully built google-meridian
  Attempting uninstall: natsort
    Found existing installation: natsort 8.4.0

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az

import IPython

from meridian import constants
from meridian.data import load
from meridian.data import test_utils
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.analysis import optimizer
from meridian.analysis import analyzer
from meridian.analysis import visualizer
from meridian.analysis import summarizer
from meridian.analysis import formatter

# check if GPU is available
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("Num CPUs Available: ", len(tf.config.experimental.list_physical_devices('CPU')))

Your runtime has 13.6 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


In [6]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

gcs_file_path = '/content/MMM1.csv'

# Keep on_bad_lines='skip' in case there are still parsing issues from previous attempts
data = pd.read_csv(gcs_file_path, on_bad_lines='skip')
print(data.columns)

# Identify columns that contain spend-related keywords (e.g., 'spend', 'cost', 'budget')
media = [
    col for col in data.columns
    if any(keyword in col.lower() for keyword in ['spend', 'cost', 'budget'])
]

# Exclude media channels with zero total spend
media = [col for col in media if data[col].sum() > 0]


print("Selected Media Channels:", media)

impressions =  [
    col for col in data.columns
    if "impression" in col.lower() or "impresseion" in col.lower()
]

print("Impressions List:", impressions)


# Convert 'date' column to datetime objects using the specified format yyyy/mm/dd
try:
    data['date'] = pd.to_datetime(data['date'], format='%Y/%m/%d', errors='coerce')
except Exception as e:
    print(f"Could not convert 'date' to datetime: {e}")

# Drop rows where date conversion failed (i.e., 'date' is NaT)
data = data.dropna(subset=['date'])

# Identify the output (revenue) column(s)
output = [
    col for col in data.columns
    if 'revenue' in col.lower()
]

# Create a time index t
data['t'] = range(len(data))

# Separate features (X) and target (y)
X = data.drop(output, axis=1)
y = data[output[0]]  # if there is only one revenue column


display(data.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/MMM1.csv'

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ----------------------------
# 1. Spend Share Analysis
# ----------------------------
print("******** Spend Share Analysis ********")
if media:
    # Calculate total spend across all identified media channels.
    total_media_spend = data[media].sum().sum()
    # Compute spend share for each media channel.
    spend_share = data[media].sum() / total_media_spend
    spend_share_df = spend_share.reset_index()
    spend_share_df.columns = ['Media_Channel', 'Spend_Share']
    print(spend_share_df)

    # Plot a bar chart of spend share.
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Spend_Share', y='Media_Channel',
                data=spend_share_df.sort_values(by='Spend_Share', ascending=False))
    plt.title("Spend Share by Media Channel")
    plt.xlabel("Spend Share")
    plt.ylabel("Media Channel")
    plt.show()
else:
    print("No media channels available for spend share analysis.")


# ----------------------------
# 2. Correlation Analysis between Inputs and Output (ecommerce_revenue)
# ----------------------------
print("\n******** Correlation Analysis ********")
revenue_col = output[0] if output and output[0] in data.columns else 'ecommerce_revenue'
print("Using revenue column:", revenue_col)

numeric_data = data.select_dtypes(include=[np.number])
corr_matrix = numeric_data.corr()

if revenue_col in corr_matrix.columns:
    corr_with_rev = corr_matrix[revenue_col].drop(labels=[revenue_col]).sort_values(ascending=False)
    print("Correlation of inputs with {}:".format(revenue_col))
    print(corr_with_rev)

    # Identify top 10 features most correlated (by absolute value) with the revenue.
    top_features = corr_with_rev.abs().sort_values(ascending=False).head(10).index.tolist() + [revenue_col]

    # Plot a heatmap displaying correlations among these top features.
    plt.figure(figsize=(10, 8))
    sns.heatmap(numeric_data[top_features].corr(), annot=True, cmap='coolwarm', center=0)
    plt.title("Heatmap of Top Correlated Features with {}".format(revenue_col))
    plt.show()
else:
    print("Revenue column not found in the correlation matrix.")


# ----------------------------
# 3. Multicollinearity Analysis (Variance Inflation Factor)
# ----------------------------
print("\n******** Multicollinearity Analysis ********")
# Use the feature set X defined earlier; select only numeric columns.
X_numeric = X.select_dtypes(include=[np.number]).drop(columns=['t'], errors='ignore')
if not X_numeric.empty:
    vif_df = pd.DataFrame()
    vif_df["Feature"] = X_numeric.columns
    # Calculate VIF for each feature.
    vif_df["VIF"] = [variance_inflation_factor(X_numeric.values, i)
                     for i in range(X_numeric.shape[1])]
    print(vif_df.sort_values(by="VIF", ascending=False))
else:
    print("No numeric features available in X for multicollinearity analysis.")


# ----------------------------
# 4. Check for Variables with Less Than 15 Records
# ----------------------------
print("\n******** Variables with Less Than 15 Records ********")
low_record_columns = {col: data[col].count() for col in data.columns if data[col].count() < 15}
if low_record_columns:
    for col, cnt in low_record_columns.items():
        print(f"Column '{col}' has only {cnt} recorded non-null entries.")
else:
    print("All variables have at least 15 records.")


In [ ]:
def create_channel_mappings(media, impressions):
    """
    Generate mapping dictionaries for media spend and impressions columns.

    Parameters:
        media (list of str): List of column names with spend/cost data.
                             Expected to have columns ending with '_spend'. # Reverted docstring to original expectation
        impressions (list of str): List of column names with impressions data.
                                   Expected to have columns ending with 'impression' or 'impresseion'. # Updated docstring
                                   Note: If this list contains spend columns (as in the original call),
                                   the resulting 'impressions_mapping' might not be meaningful for impression data.

    Returns:
        tuple: Two dictionaries:
            - correct_media_spend_to_channel: Mapping of spend/cost column names to simplified channel names.
            - correct_media_to_channel: Mapping of impressions column names to simplified channel names.
    """
    correct_media_spend_to_channel = {}
    for col in media:
        if col.endswith("spend"):
            channel_name = col[:-len("spend")]
        else:
            channel_name = col
        correct_media_spend_to_channel[col] = channel_name

    correct_media_to_channel = {}
    for col in impressions:
        col_lower = col.lower()
        if col_lower.endswith("impression"):
            channel_name = col[:-len("impression")]
        elif col_lower.endswith("impresseion"):
            channel_name = col[:-len("impresseion")]
        else:
            channel_name = col
        correct_media_to_channel[col] = channel_name

    return correct_media_spend_to_channel, correct_media_to_channel

# Correctly call create_channel_mappings passing media and impressions lists
cost_mapping, impressions_mapping = create_channel_mappings(media, impressions)

print("Cost Mapping:", cost_mapping)
print("Impressions Mapping:", impressions_mapping)

In [ ]:
skip_seasonality = True

if not skip_seasonality:
  def create_seasonality_features(data: pd.DataFrame, date_column: str, output_variable: str, yearly_seasonality: int) -> pd.DataFrame:
    """
    Creates seasonality effect features for modeling based on the yearly seasonality using the statsmodels library.
    Handles ValueError by using additive seasonality.

    Parameters:
    - data (pd.DataFrame): The input dataframe.
    - date_column (str): The name of the date column.
    - output_variable (str): The name of the output variable.
    - yearly_seasonality (int): The yearly seasonality period (e.g., 52 for weekly, 365 for daily).

    Returns:
    - pd.DataFrame: The dataframe with added seasonality features.
    """

    # Ensure the date column is in datetime format
    data[date_column] = pd.to_datetime(data[date_column])

    # Sort the data by date
    data = data.sort_values(by=date_column)

    # Set the date column as index for seasonal decomposition
    data.set_index(date_column, inplace=True)

    # try multiplicative if not then additive
    try:
        decomposition = seasonal_decompose(data[output_variable], model='multiplicative', period=yearly_seasonality)
    except ValueError as e:
        print(f"Multiplicative decomposition failed: {e}. Switching to additive model.")
        decomposition = seasonal_decompose(data[output_variable], model='additive', period=yearly_seasonality)


    # Add seasonal component as a feature
    data[f'seasonal_{output_variable}'] = decomposition.seasonal

    # Reset index to retain the date column
    data.reset_index(inplace=True)

    return data

# Example usage
# Use the actual output column name identified from the data (stored in output[0])
  data = create_seasonality_features(
    data,
    date_column='date',
    output_variable=output[0],  # Modified to use the actual revenue column name
    yearly_seasonality=52
  )

  data.head()

else:
    print("Skipping seasonality feature creation due to insufficient data.")



In [ ]:
from meridian.data import load
import numpy as np # Import numpy for array comparison

print(data.columns)

# Detect controls column dynamically
controls_candidates = ['Offline Discount']
controls = [c for c in controls_candidates if c in data.columns]

coord_to_columns = load.CoordToColumns(
    time='date',
    kpi=output[0],
    controls=controls if controls else None,
    media=impressions,
    media_spend=media,
)

# Add a check to compare impressions list and impressions_mapping keys
print("\nComparing impressions list and impressions_mapping keys:")
print("Impressions list:", impressions)
print("Impressions mapping keys:", list(impressions_mapping.keys()))

# Check if keys match the impressions list
if np.array_equal(sorted(impressions), sorted(list(impressions_mapping.keys()))):
    print("Impressions list and impressions_mapping keys match.")
else:
    print("Impressions list and impressions_mapping keys DO NOT match.")


loader = load.DataFrameDataLoader(
    df=data, # Use the data variable
    kpi_type='revenue',
    coord_to_columns=coord_to_columns,
    media_spend_to_channel=cost_mapping,
    media_to_channel=impressions_mapping, # Add the impressions mapping
)
data_meridian = loader.load()

In [ ]:
data_meridian.as_dataset()

In [ ]:
#functiom to translate the media ROI into a log normal distribution usable by the model prios

def estimate_lognormal_dist(mean, std):
    """
    Reparameterizes the LogNormal distribution in terms of its mean and std.
    Returns mu_log and std_log which can be used to define a LogNormal.
    """
    mu_log = np.log(mean) - 0.5 * np.log((std/mean)**2 + 1)
    std_log = np.sqrt(np.log((std/mean)**2 + 1))
    return mu_log, std_log

roi_mu= 2
roi_sigma= 1.5

roi_mu_log, roi_sigma_log=estimate_lognormal_dist(roi_mu, roi_sigma)
print(roi_mu_log, roi_sigma_log)
"""
# setting priors based on spend share
total_spend= data[media].sum().sum()
priors={'mu':[], 'sigma':[]}
for i in media:
  spend_share=(data[i].sum()/total_spend)*2
  roi_sigma=roi_mu*spend_share
  mu , sigma= estimate_lognormal_dist(roi_mu, roi_sigma)
  priors['mu'].append(mu)
  priors['sigma'].append(sigma)

print(priors)

roi_mu=priors['mu']
roi_sigma=priors['sigma']
print(roi_mu)"""


In [ ]:
# Prior Distribution
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
       loc=np.float32(roi_mu_log),
       scale=np.float32(roi_sigma_log),
       name=constants.ROI_M)
)

# Build Meridian Model
# Initializing ModelSpec directly with parameters from the old create_model_spec call and the prior
# Mapping n_lags and adstock_max_lag_weight to max_lag (assuming this is the correct parameter name)
model_spec = spec.ModelSpec(
    prior=prior,
    max_lag=7, # Assuming adstock_max_lag_weight maps to max_lag
)

# Initialize the model for posterior sampling
mmm = model.Meridian(input_data=data_meridian, model_spec=model_spec)

In [ ]:
from tqdm import tqdm
mmm.sample_prior(100)

mmm.sample_posterior(
    n_chains=5,
    n_adapt=1000,
    n_burnin=1000,
    n_keep=2000,
)

model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.predictive_accuracy_table()



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
parameters_to_plot=["roi_m"]
for params in parameters_to_plot:
  az.plot_trace(
      mmm.inference_data,
      var_names=params,
      compact=False,
      backend_kwargs={"constrained_layout": True},
  )


In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)

filepath = '/content/drive/MyDrive'
start_date = data['date'].min().strftime('%Y-%m-%d')
end_date = data['date'].max().strftime('%Y-%m-%d')
mmm_summarizer.output_model_results_summary('summary_output.html', filepath, start_date, end_date)
IPython.display.HTML(filename='/content/drive/MyDrive/summary_output.html')


In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()


In [ ]:
media_summary = visualizer.MediaSummary(mmm)
media_summary.summary_table()


In [ ]:

# Generate each individual chart and remove their individual config attributes to avoid conflicts on concatenation
media_summary.plot_contribution_waterfall_chart()


In [ ]:
media_summary.plot_contribution_pie_chart()


In [ ]:
media_summary.plot_spend_vs_contribution()


In [ ]:
media_summary.plot_roi_bar_chart()


In [ ]:
media_summary.plot_roi_vs_effectiveness()


In [ ]:
media_summary.plot_roi_vs_mroi()


In [ ]:
media_effects = visualizer.MediaEffects(mmm)
media_effects.plot_response_curves()

In [ ]:
media_effects.plot_adstock_decay()


In [ ]:

# Calculate total media spend in the optimization period to help set constraints
# Assuming optimization period ('2024-01-01', '2024-01-30') from original code
start_date_opt = data['date'].min().strftime('%Y-%m-%d')
end_date_opt = data['date'].max().strftime('%Y-%m-%d')
optimization_period_data = data[(data['date'] >= start_date_opt) & (data['date'] <= end_date_opt)]
current_spend_opt_period = optimization_period_data[media].sum()
print("\nCurrent spend in optimization period:")
print(current_spend_opt_period)

# Budget = total actual spend in period; constraints applied uniformly across all channels
total_budget = int(current_spend_opt_period.sum())
n_channels = len(media)
spend_constraint_lower = [0.5] * n_channels  # allow down to 50% of current spend per channel
spend_constraint_upper = [1.5] * n_channels  # allow up to 150% of current spend per channel

budget_optimizer = optimizer.BudgetOptimizer(mmm)

optimization_results = budget_optimizer.optimize(
    start_date=start_date_opt,
    end_date=end_date_opt,
    budget=total_budget,
    spend_constraint_lower=spend_constraint_lower,
    spend_constraint_upper=spend_constraint_upper,
)

In [ ]:
optimization_results.nonoptimized_data

In [ ]:
optimization_results.optimized_data


In [ ]:
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

In [ ]:
filepath = '/content/drive/MyDrive'
optimization_results.output_optimization_summary('optimization_output.html', filepath)
IPython.display.HTML(filename='/content/drive/MyDrive/optimization_output.html')